In [14]:
import os
import sys
from pathlib import Path

CONDA_PREFIX = Path(sys.executable).parents[1]

for candidate in [
    CONDA_PREFIX / "share" / "proj",
    CONDA_PREFIX / "Library" / "share" / "proj",
]:
    if (candidate / "proj.db").exists():
        PROJ_DIR = candidate
        break
else:
    raise FileNotFoundError(f"No proj.db found under {CONDA_PREFIX}")

os.environ["PROJ_LIB"] = str(PROJ_DIR)
os.environ["PROJ_DATA"] = str(PROJ_DIR)
os.environ["GDAL_DATA"] = str(CONDA_PREFIX / "share" / "gdal")

print("PROJ_DATA =", os.environ["PROJ_DATA"])
print("GDAL_DATA =", os.environ["GDAL_DATA"])

PROJ_DATA = /home/ajkerr1/.conda/envs/graha-lunar-fm/share/proj
GDAL_DATA = /home/ajkerr1/.conda/envs/graha-lunar-fm/share/gdal


# LFM Segmentation Fine-Tuning Without TerraTorch CLI

This notebook builds the TerraMind/TerraTorch fine-tuning stack directly from Python classes. It avoids `terratorch fit` and `build_lightning_cli`, while preserving the same model factory path used by the YAML configs.

The pretrained backbone weights are loaded through `LunarBackbone` via `model_args["backbone_checkpoint_path"]`.

In [15]:
from pathlib import Path
import os
import sys

NOTEBOOK_DIR = Path.cwd().resolve()
LFM_ROOT = NOTEBOOK_DIR.parents[1]
REPO_ROOT = LFM_ROOT / "lfm" / "full_model" / "graha-lunar-fm"

# Pretraining artifact directory used by the imported Graha/Lunar-FM code.
PRETRAIN_DIR = Path(
    "/explore/nobackup/projects/lfm/gabby/Lunar-FM/experiments/"
    "lunarfm_base_dual_full_nas_no_nans_256_256_lr1e-4_wd0.05"
).resolve()

BACKBONE_WEIGHTS = PRETRAIN_DIR / "checkpoints/checkpoint_weights_final.pt"
BACKBONE_CFG = PRETRAIN_DIR / "full_config.yaml"
MODALITY_INFO = PRETRAIN_DIR / "modality_info.yaml"

DATA_ROOT = NOTEBOOK_DIR / "data"
BASE_OUTPUT_DIR = NOTEBOOK_DIR / "outputs" / "graha_finetuning"

# The datamodule z-scores WAC with fine-tuning train-split stats.
# This range is TerraMind modality metadata for normalized tensors,
# not the raw WAC .tif min/max range.
NORMALIZED_WAC_DATA_RANGE = [-1.0, 1.0]

for import_path in [REPO_ROOT, LFM_ROOT]:
    if str(import_path) not in sys.path:
        sys.path.insert(0, str(import_path))

print("Notebook directory:", NOTEBOOK_DIR)
print("Graha/Lunar-FM code root:", REPO_ROOT)
print("Data root:", DATA_ROOT)
print("Backbone weights:", BACKBONE_WEIGHTS)
print("Backbone config:", BACKBONE_CFG)
print("Modality info:", MODALITY_INFO)
print("Base output directory:", BASE_OUTPUT_DIR)
print("Normalized WAC modality data_range:", NORMALIZED_WAC_DATA_RANGE)

Notebook directory: /panfs/ccds02/nobackup/people/ajkerr1/Lunar_FM/full_model_lfm/lfm/notebooks/graha-flm-finetuning
Graha/Lunar-FM code root: /panfs/ccds02/nobackup/people/ajkerr1/Lunar_FM/full_model_lfm/lfm/graha-lunar-fm
Data root: /panfs/ccds02/nobackup/people/ajkerr1/Lunar_FM/full_model_lfm/lfm/notebooks/graha-flm-finetuning/data
Backbone weights: /panfs/ccds02/nobackup/projects/lfm/gabby/Lunar-FM/experiments/lunarfm_base_dual_full_nas_no_nans_256_256_lr1e-4_wd0.05/checkpoints/checkpoint_weights_final.pt
Backbone config: /panfs/ccds02/nobackup/projects/lfm/gabby/Lunar-FM/experiments/lunarfm_base_dual_full_nas_no_nans_256_256_lr1e-4_wd0.05/full_config.yaml
Modality info: /panfs/ccds02/nobackup/projects/lfm/gabby/Lunar-FM/experiments/lunarfm_base_dual_full_nas_no_nans_256_256_lr1e-4_wd0.05/modality_info.yaml
Base output directory: /panfs/ccds02/nobackup/people/ajkerr1/Lunar_FM/full_model_lfm/lfm/notebooks/graha-flm-finetuning/outputs/graha_finetuning
Normalized WAC modality data_ran

In [16]:
required_paths = [
    REPO_ROOT,
    BACKBONE_WEIGHTS,
    BACKBONE_CFG,
    MODALITY_INFO,
    DATA_ROOT,
    DATA_ROOT / "train" / "chips",
    DATA_ROOT / "train" / "labels",
    DATA_ROOT / "val" / "chips",
    DATA_ROOT / "val" / "labels",
]

missing = [path for path in required_paths if not path.exists()]
if missing:
    raise FileNotFoundError("Missing required paths:\n" + "\n".join(str(path) for path in missing))

In [17]:
import torch
from lightning.pytorch import Trainer, seed_everything
from lightning.pytorch.callbacks import LearningRateMonitor, ModelCheckpoint
from lightning.pytorch.loggers import TensorBoardLogger

# Importing terratorch_integration triggers registration of:
# - lunarmind_v1_* backbones
# - custom necks
# - Lunar segmentation/object-detection tasks
import terratorch_integration  # noqa: F401
from terratorch_integration.lunar_segmentation_task import LunarShapeSegmentationTask

from lfm.full_model.datamodules import LunarSemanticSegmentationDatamodule
from lfm.full_model.utils import ValidationPlotCallback, plot_validation_predictions
from lfm.full_model.utils import create_timestamped_output_dir


class NotebookLunarShapeSegmentationTask(LunarShapeSegmentationTask):
    """Drop datamodule metadata that should not be forwarded to the model."""

    def _drop_extra_batch_metadata(self, batch):
        if isinstance(batch, dict):
            batch.pop("num_craters", None)
        return batch

    def training_step(self, batch, *args, **kwargs):
        return super().training_step(self._drop_extra_batch_metadata(batch), *args, **kwargs)

    def validation_step(self, batch, *args, **kwargs):
        return super().validation_step(self._drop_extra_batch_metadata(batch), *args, **kwargs)

    def test_step(self, batch, *args, **kwargs):
        return super().test_step(self._drop_extra_batch_metadata(batch), *args, **kwargs)

    def predict_step(self, batch, *args, **kwargs):
        return super().predict_step(self._drop_extra_batch_metadata(batch), *args, **kwargs)

In [18]:
OUTPUT_DIR = create_timestamped_output_dir(BASE_OUTPUT_DIR)
PLOTS_DIR = OUTPUT_DIR / "plots"
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

print("Output directory:", OUTPUT_DIR)
print("Plots directory:", PLOTS_DIR)

Using output subdir: /panfs/ccds02/nobackup/people/ajkerr1/Lunar_FM/full_model_lfm/lfm/notebooks/graha-flm-finetuning/outputs/graha_finetuning/date_2026_07_15-time_15_12_01
Output directory: /panfs/ccds02/nobackup/people/ajkerr1/Lunar_FM/full_model_lfm/lfm/notebooks/graha-flm-finetuning/outputs/graha_finetuning/date_2026_07_15-time_15_12_01
Plots directory: /panfs/ccds02/nobackup/people/ajkerr1/Lunar_FM/full_model_lfm/lfm/notebooks/graha-flm-finetuning/outputs/graha_finetuning/date_2026_07_15-time_15_12_01/plots


In [ ]:
max_epochs=MAX_EPOCHS

In [19]:
seed_everything(42)

COMMON_DATAMODULE_ARGS = dict(
    data_root=DATA_ROOT,
    crop_size=256,
    image_glob="*.tif",
    label_glob="*_label.*",
    image_suffix="_input_wac_static_chip",
    label_suffix="_label",
    no_data_replace=0.0,
    no_label_replace=None,
)

stats_datamodule = LunarSemanticSegmentationDatamodule(
    **COMMON_DATAMODULE_ARGS,
    batch_size=16,
    num_workers=10,
    # Leave stats unset here so batches contain cropped, unnormalized WAC.
    means=None,
    stds=None,
)
stats_datamodule.setup("fit")

n_pixels = 0
sum_x = None
sum_x2 = None

for batch in stats_datamodule.train_dataloader():
    x = batch["image"]  # B,C,H,W, unnormalized
    b, c, h, w = x.shape
    batch_pixels = b * h * w
    x_sum = x.sum(dim=(0, 2, 3))
    x2_sum = (x * x).sum(dim=(0, 2, 3))
    sum_x = x_sum if sum_x is None else sum_x + x_sum
    sum_x2 = x2_sum if sum_x2 is None else sum_x2 + x2_sum
    n_pixels += batch_pixels

means_tensor = sum_x / n_pixels
stds_tensor = torch.sqrt(torch.clamp(sum_x2 / n_pixels - means_tensor ** 2, min=1e-12))
means = means_tensor.tolist()
stds = stds_tensor.tolist()

print("per-band means:", means)
print("per-band stds:", stds)

datamodule = LunarSemanticSegmentationDatamodule(
    **COMMON_DATAMODULE_ARGS,
    batch_size=16,
    num_workers=10,
    # Apply per-band z-score normalization from the fine-tuning train split.
    means=means,
    stds=stds,
)

[rank: 0] Seed set to 42


per-band means: [0.023870304226875305, 0.03239678964018822, 0.0345771387219429, 0.036306023597717285, 0.03883824869990349, 0.015070014633238316, 0.017845287919044495]
per-band stds: [0.014905835501849651, 0.019833091646432877, 0.02095038630068302, 0.02188112959265709, 0.023320039734244347, 0.008971190080046654, 0.010731484740972519]


In [20]:
datamodule.setup("fit")
sample_batch = next(iter(datamodule.train_dataloader()))

WAC_MODALITY = "wac"
WAC_NUM_CHANNELS = int(sample_batch["image"].shape[1])

print("batch keys:", sample_batch.keys())
print("image:", tuple(sample_batch["image"].shape), sample_batch["image"].dtype)
print("batch image per-band mean:", sample_batch["image"].mean(dim=(0, 2, 3)).tolist())
print("batch image per-band std:", sample_batch["image"].std(dim=(0, 2, 3)).tolist())
print("mask:", tuple(sample_batch["mask"].shape), sample_batch["mask"].dtype)
print("mask values:", torch.unique(sample_batch["mask"]).tolist())
print("WAC channels registered for model:", WAC_NUM_CHANNELS)
if "crater_boxes" in sample_batch:
    print("crater boxes per image:", [tuple(x.shape) for x in sample_batch["crater_boxes"]])

task = NotebookLunarShapeSegmentationTask(
    backbone_lr=5.0e-5,
    head_lr=2.0e-4,
    layer_decay=0.75,
    weight_decay=0.05,
    warmup_steps=500,
    shape_loss_weight=0.05,
    shape_loss_pad_frac=0.3,
    model_factory="EncoderDecoderFactory",
    model_args={
        "backbone": "lunarmind_v1_base",
        "backbone_checkpoint_path": str(BACKBONE_WEIGHTS),
        "backbone_cfg": str(BACKBONE_CFG),
        "backbone_modality_info_path": str(MODALITY_INFO),
        "backbone_modalities": [WAC_MODALITY],
        "backbone_new_modalities": {
            WAC_MODALITY: {
                "type": "image",
                "num_channels": WAC_NUM_CHANNELS,
                "data_range": NORMALIZED_WAC_DATA_RANGE,
            },
        },
        "backbone_patch_size": 8,
        "backbone_remove_register_tokens": False,
        "backbone_merge_method": None,
        "necks": [
            {"name": "SelectIndices", "indices": [2, 5, 8, 11]},
            {"name": "ReshapeTokensToImage", "remove_cls_token": False, "h": 32},
            {"name": "LearnedInterpolateToPyramidal"},
        ],
        "decoder": "UNetDecoder",
        "decoder_channels": [512, 256, 128, 64],
        "head_channel_list": [256],
        "num_classes": 2,
        "head_dropout": 0.1,
    },
    loss="dice",
    class_names=["Background", "Crater"],
    freeze_backbone=False,
    freeze_decoder=False,
    plot_on_val=0,
)

batch keys: dict_keys(['image', 'mask', 'filename'])
image: (16, 7, 256, 256) torch.float32
batch image per-band mean: [0.2652943432331085, 0.2787616550922394, 0.2865247130393982, 0.29237842559814453, 0.2913972735404968, 0.276884526014328, 0.29494062066078186]
batch image per-band std: [0.9719842672348022, 0.9833362698554993, 0.9911004900932312, 0.9977791905403137, 1.0003973245620728, 0.9899802207946777, 0.9934325218200684]
mask: (16, 256, 256) torch.int64
mask values: [0, 1]
WAC channels registered for model: 7
Auto-detected num_register_tokens=0 from checkpoint
New modality embeddings (randomly initialized): 3 keys
encoder_embeddings.aspect.mod_emb
encoder_embeddings.aspect.pos_emb
encoder_embeddings.aspect.proj.weight
encoder_embeddings.aspect_3m.mod_emb
encoder_embeddings.aspect_3m.pos_emb
encoder_embeddings.aspect_3m.proj.weight
encoder_embeddings.dtm.mod_emb
encoder_embeddings.dtm.pos_emb
encoder_embeddings.dtm.proj.weight
encoder_embeddings.dtm_3m.mod_emb
encoder_embeddings.dtm_

In [21]:
backbone = task.model.encoder
print(type(backbone))
print("modalities:", backbone.modalities)
print(f"backbone params: {backbone.get_num_params():,}")

<class 'terratorch_integration.lunar_backbone.LunarBackbone'>
modalities: ['wac']
backbone params: 199,173,888


In [22]:
datamodule.setup("fit")
batch = next(iter(datamodule.train_dataloader()))

print("batch keys:", batch.keys())
print("image:", tuple(batch["image"].shape), batch["image"].dtype)
print("mask:", tuple(batch["mask"].shape), batch["mask"].dtype)
print("mask values:", torch.unique(batch["mask"]).tolist())
if "crater_boxes" in batch:
    print("crater boxes per image:", [tuple(x.shape) for x in batch["crater_boxes"]])

batch keys: dict_keys(['image', 'mask', 'filename'])
image: (16, 7, 256, 256) torch.float32
mask: (16, 256, 256) torch.int64
mask values: [0, 1]


In [23]:
trainer = Trainer(
    accelerator="gpu" if torch.cuda.is_available() else "cpu",
    devices=1,
    precision="32",
    max_epochs=100,
    check_val_every_n_epoch=1,
    log_every_n_steps=5,
    logger=TensorBoardLogger(
        save_dir=str(OUTPUT_DIR / "tb_logs"),
        name="crater-segmentation-notebook-direct",
    ),
    callbacks=[
        LearningRateMonitor(logging_interval="epoch"),
        ValidationPlotCallback(
            output_dir=OUTPUT_DIR,
            n_samples=5,
            every_n_epochs=1,
            dpi=150,
        ),
        ModelCheckpoint(
            dirpath=str(OUTPUT_DIR / "checkpoints"),
            monitor="val/loss",
            mode="min",
            filename="best-{epoch:02d}-{val/loss:.3f}",
            save_top_k=3,
            save_last=True,
        ),
    ],
)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


In [ ]:
# Run fine-tuning when ready.
trainer.fit(task, datamodule=datamodule)

Epoch 5/99 ━━━━━━━━━━━━━━╺━━━━━━━━━━━━━━━━━━━━━━━━━ 11/31 0:00:15 • 0:00:27 0.74it/s v_num: 0.000


Detected KeyboardInterrupt, attempting graceful shutdown ...
